In [0]:
CONNECTION = "googledrive_con"
SOURCE_URL = "https://docs.google.com/spreadsheets/d/1s79gryMvLF6I0jxRYkXiDYZVqkiiSn04S6zNomeM3Jg/edit?usp=drive_link"  

CATALOG, SCHEMA = "bronze", "taxonomy"
VOLUME = "_checkpoints"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

sheets = {"dimPlant": "dimPlant", "dimMaterial": "dimMaterial"}  # sheet name -> table name

queries = []
for sheet, table in sheets.items():
    base = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{table}"

    stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "excel")
        .option("databricks.connection", CONNECTION)
        .option("cloudFiles.schemaLocation", f"{base}/schema")
        .option("cloudFiles.inferColumnTypes", True)
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("headerRows", 1)
        .option("dataAddress", f"'{sheet}'!A1")
        .load(SOURCE_URL)
        .selectExpr("*", "_metadata.file_name AS _source_file", "current_timestamp() AS _ingested_at")
    )

    q = (
        stream.writeStream
        .option("checkpointLocation", f"{base}/checkpoint")
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.{SCHEMA}.{table}")
    )
    queries.append(q)

for q in queries:
    q.awaitTermination()
